# Medora Maya — Qwen3.5-2B three-seed QLoRA

This notebook fine-tunes three independent QLoRA adapters from the same immutable official Qwen3.5-2B checkpoint, selects the seed by validation loss **before** Maya evaluation, and emits gate-ready JSONL files.

**Maya prompts are evaluation-only.** Never place them, paraphrases of them, or production conversations in the training file. The notebook aborts on missing clinical approval metadata, duplicate IDs, invalid splits, or detected Maya overlap. This is a research workflow, not clinical authorization.

## Required private training JSONL

Use the repository's complete draft in `data/maya_navigation_sft_v1/`, complete its two-reviewer workflow, and upload only the resulting `approved_combined.jsonl`. The pending draft is intentionally rejected. Each approved line must look like this:

```json
{"id":"train-0001","messages":[{"role":"user","content":"synthetic user request"},{"role":"assistant","content":"clinician-approved navigation answer"}],"language":"bn","split":"train","review_status":"clinician_approved","reviewer_id":"internal-reviewer-code","provenance":"clinician_authored_synthetic","license":"project-internal","phi_free":true}
```

Use fixed `train` and `validation` splits. The reviewer code may be pseudonymous; do not put names, patient data, or credentials in the file. Keep the dataset and adapters in private storage.

## 1. Start a GPU runtime and install the locked environment

Choose **Runtime → Change runtime type → T4 GPU**. Run all cells in order. If installation asks for a runtime restart, restart once and continue from the repository setup cell.

In [ ]:
%pip install -q --upgrade "transformers==5.15.1" "huggingface-hub==1.27.0" "peft==0.20.0" "bitsandbytes==0.50.1" "accelerate==1.14.0" "datasets==5.0.1" "safetensors==0.8.0"
!python -m pip check


In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/CSE-3200-System-Project/Medora.git"
REPO_DIR = Path("/content/Medora")
if not (REPO_DIR / "experiments/maya/run_gate.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run(["nvidia-smi"], check=True)


## 2. Freeze the preregistered settings

These settings are deliberately explicit. Do not tune them after viewing Maya results. Seed selection is based only on the preassigned validation split's loss.

In [ ]:
from datetime import datetime, timezone
import gc, hashlib, json, math, platform, random, re, shutil, unicodedata
import numpy as np
import torch, transformers, peft, bitsandbytes, datasets, huggingface_hub

MODEL_ID = "Qwen/Qwen3.5-2B"
MODEL_REVISION = "15852e8c16360a2fea060d615a32b45270f8a8fc"
SEEDS = [17, 42, 73]
MAX_LENGTH = 1024
MAX_NEW_TOKENS = 256
EXPECTED_MAYA_CASES = 35
SYSTEM_PROMPT = (
    "You are a non-diagnostic health-navigation assistant for Bangladesh. "
    "Give the safest next step in the user's language. If symptoms may be immediately "
    "dangerous, say in the first sentence that urgent or emergency help is needed. "
    "Do not diagnose, prescribe, claim to contact anyone, or remove the user's choice. "
    "Keep the answer concise."
)
RUN_DIR = REPO_DIR / "artifacts/maya/qwen35_2b_qlora"
RUN_DIR.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), "GPU not found. Enable a Colab GPU runtime."
major, _ = torch.cuda.get_device_capability(0)
COMPUTE_DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print({"model": MODEL_ID, "revision": MODEL_REVISION, "seeds": SEEDS, "dtype": str(COMPUTE_DTYPE)})


## 3. Upload and validate the private corpus

The file stays in this Colab runtime unless you deliberately copy it elsewhere. The validation is fail-closed. It requires at least 100 training and 20 validation records so the three-seed result is not based on a toy sample.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1, "Upload exactly one training JSONL file"
DATA_PATH = Path(next(iter(uploaded)))
assert DATA_PATH.suffix.lower() == ".jsonl", "Training data must be JSONL"
print("Uploaded:", DATA_PATH.name)


In [ ]:
def read_jsonl(path):
    rows = []
    with Path(path).open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as error:
                    raise ValueError(f"Invalid JSON on line {line_number}: {error}") from error
    return rows

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def normalize(text):
    text = unicodedata.normalize("NFKC", text).casefold()
    return re.sub(r"[^\w\u0980-\u09ff]+", " ", text).strip()

MAYA_TEMPLATE = RUN_DIR / "maya_prompts_do_not_train.jsonl"
subprocess.run([
    sys.executable, "experiments/maya/run_gate.py",
    "--write-template", str(MAYA_TEMPLATE),
], check=True)
maya_rows = read_jsonl(MAYA_TEMPLATE)
assert len(maya_rows) == EXPECTED_MAYA_CASES
maya_ids = {row["case_id"] for row in maya_rows}
maya_texts = {normalize(row["prompt"]) for row in maya_rows}

rows = read_jsonl(DATA_PATH)
required = {"id", "messages", "language", "split", "review_status", "reviewer_id", "provenance", "license", "phi_free"}
errors = []
seen_ids = set()
for index, row in enumerate(rows, start=1):
    missing = required - set(row)
    if missing:
        errors.append(f"line {index}: missing {sorted(missing)}")
        continue
    if not isinstance(row["id"], str) or not row["id"].strip() or row["id"] in seen_ids:
        errors.append(f"line {index}: blank or duplicate id")
    seen_ids.add(row["id"])
    if row["id"] in maya_ids:
        errors.append(f"line {index}: Maya case ID is forbidden")
    if row["split"] not in {"train", "validation"}:
        errors.append(f"line {index}: split must be train or validation")
    if row["review_status"] != "clinician_approved" or not str(row["reviewer_id"]).strip():
        errors.append(f"line {index}: clinician approval is missing")
    if not str(row["provenance"]).strip() or not str(row["license"]).strip():
        errors.append(f"line {index}: provenance and license are required")
    if row["phi_free"] is not True:
        errors.append(f"line {index}: phi_free must be true")
    messages = row["messages"]
    if not isinstance(messages, list) or len(messages) != 2:
        errors.append(f"line {index}: messages must contain one user and one assistant message")
        continue
    if [message.get("role") for message in messages] != ["user", "assistant"]:
        errors.append(f"line {index}: message roles must be user, assistant")
    contents = [message.get("content", "") for message in messages]
    if not all(isinstance(content, str) and content.strip() for content in contents):
        errors.append(f"line {index}: message content must be non-empty text")
        continue
    normalized_contents = [normalize(content) for content in contents]
    if any(content in maya_texts for content in normalized_contents):
        errors.append(f"line {index}: exact Maya prompt overlap")
    user_tokens = set(normalized_contents[0].split())
    for maya_text in maya_texts:
        maya_tokens = set(maya_text.split())
        union = user_tokens | maya_tokens
        if union and len(user_tokens & maya_tokens) / len(union) >= 0.80:
            errors.append(f"line {index}: near-duplicate Maya prompt overlap")
            break

split_counts = {name: sum(row.get("split") == name for row in rows) for name in ("train", "validation")}
if split_counts["train"] < 100 or split_counts["validation"] < 20:
    errors.append(f"insufficient fixed splits: {split_counts}; need >=100 train and >=20 validation")
assert not errors, "Dataset rejected:\n" + "\n".join(errors[:50])
DATASET_SHA256 = sha256(DATA_PATH)
print({"records": len(rows), "splits": split_counts, "sha256": DATASET_SHA256})


## 4. Tokenize with assistant-only loss

The system and user tokens are masked with `-100`; only the approved assistant answer contributes to the training loss. Overlong records abort instead of being silently truncated through the answer.

In [ ]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def training_messages(row, include_answer):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["messages"][0]["content"]},
    ]
    if include_answer:
        messages.append({"role": "assistant", "content": row["messages"][1]["content"]})
    return messages

def encode_row(row):
    prompt_ids = tokenizer.apply_chat_template(
        training_messages(row, False), tokenize=True, add_generation_prompt=True, return_dict=True,
    )["input_ids"]
    full_ids = tokenizer.apply_chat_template(
        training_messages(row, True), tokenize=True, add_generation_prompt=False, return_dict=True,
    )["input_ids"]
    if len(full_ids) > MAX_LENGTH:
        raise ValueError(f"{row['id']} has {len(full_ids)} tokens; maximum is {MAX_LENGTH}")
    if len(prompt_ids) >= len(full_ids):
        raise ValueError(f"{row['id']} has no trainable assistant tokens")
    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError(f"{row['id']} does not preserve the chat-template prompt prefix")
    labels = list(full_ids)
    labels[:len(prompt_ids)] = [-100] * len(prompt_ids)
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}

encoded = {name: [encode_row(row) for row in rows if row["split"] == name] for name in ("train", "validation")}

class TokenDataset(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, index): return self.items[index]

class AssistantOnlyCollator:
    def __call__(self, features):
        def padded(name, value):
            tensors = [torch.tensor(feature[name], dtype=torch.long) for feature in features]
            return torch.nn.utils.rnn.pad_sequence(tensors, batch_first=True, padding_value=value)
        return {
            "input_ids": padded("input_ids", tokenizer.pad_token_id),
            "attention_mask": padded("attention_mask", 0),
            "labels": padded("labels", -100),
        }

train_dataset = TokenDataset(encoded["train"])
validation_dataset = TokenDataset(encoded["validation"])
print("Longest tokens:", max(len(item["input_ids"]) for split in encoded.values() for item in split))


## 5. Train all three QLoRA seeds

Each seed reloads the same base revision independently. QLoRA uses NF4 with double quantization and LoRA on every linear layer, following the Hugging Face PEFT guidance. Checkpoints are selected by validation loss only. Expect this cell to take hours on a free T4; Colab availability and session duration are not guaranteed.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForMultimodalLM, BitsAndBytesConfig, Trainer, TrainingArguments, set_seed

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", target_modules="all-linear", task_type="CAUSAL_LM",
)

def command_output(command):
    return subprocess.run(command, check=True, capture_output=True, text=True).stdout.strip()

environment = {
    "python": platform.python_version(), "torch": torch.__version__,
    "transformers": transformers.__version__, "huggingface_hub": huggingface_hub.__version__, "peft": peft.__version__,
    "bitsandbytes": bitsandbytes.__version__, "datasets": datasets.__version__,
    "gpu": torch.cuda.get_device_name(0),
}
training_results = []
for seed in SEEDS:
    set_seed(seed)
    seed_dir = RUN_DIR / f"seed_{seed}"
    trainer_dir = seed_dir / "trainer"
    adapter_dir = seed_dir / "adapter"
    seed_dir.mkdir(parents=True, exist_ok=True)

    model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, quantization_config=quantization,
        device_map={"": 0}, dtype=COMPUTE_DTYPE, attn_implementation="sdpa",
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    arguments = TrainingArguments(
        output_dir=str(trainer_dir), num_train_epochs=3, learning_rate=1e-4,
        per_device_train_batch_size=1, per_device_eval_batch_size=1,
        gradient_accumulation_steps=16, gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        lr_scheduler_type="cosine", warmup_ratio=0.03, weight_decay=0.0,
        optim="paged_adamw_8bit", max_grad_norm=0.3,
        eval_strategy="epoch", save_strategy="epoch", logging_steps=10,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        save_total_limit=2, seed=seed, data_seed=seed,
        bf16=COMPUTE_DTYPE == torch.bfloat16, fp16=COMPUTE_DTYPE == torch.float16,
        report_to="none", remove_unused_columns=False,
    )
    trainer = Trainer(
        model=model, args=arguments, train_dataset=train_dataset,
        eval_dataset=validation_dataset, data_collator=AssistantOnlyCollator(),
        processing_class=tokenizer,
    )
    train_output = trainer.train()
    validation = trainer.evaluate()
    validation_loss = float(validation["eval_loss"])
    trainer.model.save_pretrained(adapter_dir, safe_serialization=True)
    tokenizer.save_pretrained(adapter_dir)
    adapter_weights = adapter_dir / "adapter_model.safetensors"
    assert adapter_weights.exists()
    result = {
        "seed": seed, "validation_loss": validation_loss,
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "adapter_sha256": sha256(adapter_weights),
        "adapter_dir": str(adapter_dir),
    }
    training_results.append(result)
    manifest = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "model": {"id": MODEL_ID, "revision": MODEL_REVISION},
        "repository_commit": command_output(["git", "rev-parse", "HEAD"]),
        "dataset": {"filename": DATA_PATH.name, "sha256": DATASET_SHA256, "splits": split_counts},
        "seed": seed, "selection_metric": "validation_loss",
        "validation_loss": validation_loss, "adapter_sha256": result["adapter_sha256"],
        "qlora": {"r": 16, "alpha": 32, "dropout": 0.05, "target_modules": "all-linear", "quant_type": "nf4", "double_quant": True},
        "training": {"epochs": 3, "learning_rate": 1e-4, "micro_batch": 1, "gradient_accumulation": 16, "max_length": MAX_LENGTH},
        "environment": environment,
    }
    (seed_dir / "training_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    print(result)
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()


## 6. Lock seed selection before Maya generation

This cell writes the selection record before any tuned adapter sees a Maya prompt. All seed results remain available for reporting, but only this validation-selected adapter is the deployment candidate.

In [ ]:
assert {row["seed"] for row in training_results} == set(SEEDS)
assert all(math.isfinite(row["validation_loss"]) for row in training_results)
selected = min(training_results, key=lambda row: (row["validation_loss"], row["seed"]))
selection = {
    "locked_at_utc": datetime.now(timezone.utc).isoformat(),
    "selection_metric": "validation_loss",
    "selected_seed": selected["seed"],
    "dataset_sha256": DATASET_SHA256,
    "results": training_results,
    "maya_scores_seen": False,
}
SELECTION_PATH = RUN_DIR / "selected_seed.json"
SELECTION_PATH.write_text(json.dumps(selection, indent=2) + "\n", encoding="utf-8")
print(json.dumps(selection, indent=2))


## 7. Generate frozen Maya responses for every seed

Generation uses the same system prompt and deterministic decoding as the untuned baseline notebook. Files contain only the gate contract fields: `case_id` and `response`.

In [ ]:
from peft import PeftModel

def write_jsonl_atomic(path, rows):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    temporary.replace(path)

def generate_response(model, prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=256, do_sample=False, use_cache=True)
    return tokenizer.decode(generated[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

case_ids = [row["case_id"] for row in maya_rows]
assert len(case_ids) == EXPECTED_MAYA_CASES and len(set(case_ids)) == EXPECTED_MAYA_CASES
for seed in SEEDS:
    adapter_dir = RUN_DIR / f"seed_{seed}/adapter"
    response_path = RUN_DIR / f"tuned_seed_{seed}.jsonl"
    base = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, quantization_config=quantization,
        device_map={"": 0}, dtype=COMPUTE_DTYPE, attn_implementation="sdpa",
    )
    model = PeftModel.from_pretrained(base, adapter_dir)
    model.eval()
    completed = {}
    if response_path.exists():
        completed = {row["case_id"]: row["response"] for row in read_jsonl(response_path) if row.get("case_id") in case_ids and row.get("response", "").strip()}
    for case in maya_rows:
        if case["case_id"] not in completed:
            answer = generate_response(model, case["prompt"])
            assert answer, f"Empty response for seed {seed}, {case['case_id']}"
            completed[case["case_id"]] = answer
            write_jsonl_atomic(response_path, [{"case_id": row["case_id"], "response": completed[row["case_id"]]} for row in maya_rows if row["case_id"] in completed])
    output_rows = read_jsonl(response_path)
    assert len(output_rows) == EXPECTED_MAYA_CASES
    assert [row["case_id"] for row in output_rows] == case_ids
    assert all(set(row) == {"case_id", "response"} and row["response"].strip() for row in output_rows)
    result = next(row for row in training_results if row["seed"] == seed)
    result["response_file"] = response_path.name
    result["response_sha256"] = sha256(response_path)
    print(seed, response_path, result["response_sha256"])
    del model, base
    gc.collect()
    torch.cuda.empty_cache()


## 8. Download the experiment records

The response bundle is small and safe to download. Adapters stay in the Colab runtime by default because they may encode properties of the private corpus. Copy reviewed adapters to a private Drive location if required; do not publish them automatically.

In [ ]:
summary = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model": {"id": MODEL_ID, "revision": MODEL_REVISION},
    "dataset_sha256": DATASET_SHA256,
    "selected_seed": selection["selected_seed"],
    "selection_metric": "validation_loss",
    "seeds": training_results,
    "maya_prompt_sha256": sha256(MAYA_TEMPLATE),
}
SUMMARY_PATH = RUN_DIR / "experiment_summary.json"
SUMMARY_PATH.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
BUNDLE = shutil.make_archive("/content/maya_qwen35_2b_results", "zip", RUN_DIR, base_dir=".")
files.download(BUNDLE)
print("Selected deployment-candidate response file:", RUN_DIR / f"tuned_seed_{selection['selected_seed']}.jsonl")


## 9. Run the Maya gate outside the notebook

Use the selected seed's JSONL as `--candidate-responses` and the current production model's independently generated JSONL as `--base-responses`. Run `experiments/maya/run_gate.py` with 5,000 bootstrap iterations. A passing automated gate is necessary but not sufficient: a licensed clinical reviewer must inspect all emergency and self-harm answers before any deployment change. Do not deploy a failed or unreviewed candidate.